### Installing Required Libraries

In [1]:
# Installing Libraries

!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q pymupdf
!pip install -q torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

This cell installs the external packages required by the notebook (PDF parsing, embeddings, vector store, model inference and supporting libraries).

Run this once at session start: it prepares the Python environment by fetching wheels and their dependencies.

Expected output: pip download/install logs and no import errors when later cells run.

Common issues: package installation can fail when network access, conflicting Python versions, or missing system libraries occur; if you see binary/compatibility errors (e.g., bitsandbytes or CUDA-related), restart the kernel after fixing the environment or install a CPU-only alternative.

### Import Libraies

In [2]:
# Importing Libraries

import fitz
import chromadb
import torch
import numpy as np
import pandas as pd
import textwrap

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

This cell imports all Python modules used across the notebook so subsequent code cells can call them directly.

Why: centralizing imports makes it easy to spot missing dependencies early and to understand the core libraries used (PDF reader, vector DB, ML frameworks, and utility modules).

Expected output: no exceptions; if an import fails, re-run the install cell and restart the kernel.

Troubleshooting: version mismatches (torch, transformers, bitsandbytes) or missing system libs typically cause import errors—note the error trace and re-install the appropriate wheel or adjust versions.

### Load Embedding Mode

In [3]:
# LOAD EMBEDDING MODEL

print("="*80)
print("LOADING EMBEDDING MODEL")
print("="*80)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(" Embedding Model Loaded Successfully!")

LOADING EMBEDDING MODEL


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding Model Loaded Successfully!


This cell loads the sentence embedding model used to convert text chunks into dense vectors for semantic search.

Why: embeddings are the core representation for matching queries to document chunks in the vector store.

Expected output: download progress and a ready-to-use `embedding_model`. If you see out-of-memory or download errors, consider using a smaller model or ensuring network access.

Note: embeddings are computed once for the document and reused for many queries to keep retrieval fast.

### Load Mistral Model

In [4]:
# LOAD MISTRAL MODEL

print("\n")
print("="*80)
print("LOADING MISTRAL MODEL")
print("="*80)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.float16)

print(" Mistral Model Loaded Successfully!")



LOADING MISTRAL MODEL


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

 Mistral Model Loaded Successfully!


This cell downloads and initializes the tokenizer and the Mistral causal language model used for generation.

Why: the tokenizer maps text to token IDs and the model produces the answer text; both must be loaded before creating the generation pipeline.

Expected output: model weight downloads and a model object placed on the selected device (CPU/GPU).

Common problems: large models require GPU memory; if you hit memory or device errors, either switch to a smaller model, reduce dtype precision (if supported), or run on a machine with more GPU RAM.

### Create Text Generation Pipeline

In [5]:
# CREATE TEXT GENERATION PIPELINE

print("\n")
print("CREATING GENERATION PIPELINE")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    pad_token_id=tokenizer.eos_token_id
)

print(" Pipeline Ready!")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.




CREATING GENERATION PIPELINE
 Pipeline Ready!


This cell creates a `generator` pipeline that simplifies text generation calls into one object with sensible defaults (max tokens, temperature).

Why: a pipeline provides a convenient, consistent interface for inference across the notebook.

Expected output: a ready `generator` that returns generated text when called.

Tuning tip: change `temperature` for more/less creativity and `max_new_tokens` to control response length; for deterministic answers set `temperature` to 0.

### Initialize ChromaDB



In [6]:
# INITIALIZE CHROMADB

print("\n")
print("="*80)
print("INITIALIZING CHROMADB")
print("="*80)

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="pdf_rag_collection", get_or_create=True)

print("ChromaDB Collection Created!")



INITIALIZING CHROMADB
ChromaDB Collection Created!


This cell initializes the ChromaDB client and creates (or connects to) a collection named `pdf_rag_collection` that will store document chunks and their embeddings.

Why: the vector store enables efficient nearest-neighbor search to retrieve relevant document passages for RAG-style answers.

Expected output: confirmation that the collection exists.

Notes: Chroma persists vectors and metadata; use a persistent path for production. If collection creation fails, check Chroma installation and any path/permissions issues.

### Loading Document

In [8]:
# Loading Documents

print("\n")
print("="*80)
print("LOADING PDF DOCUMENT")
print("="*80)

PDF_PATH = "/content/attention.pdf"
document = fitz.open(PDF_PATH)

print(" PDF Loaded Successfully!")
print(" Total Pages:", len(document))



LOADING PDF DOCUMENT
 PDF Loaded Successfully!
 Total Pages: 15


This cell opens the PDF file and loads it into the `document` object so the notebook can iterate and extract page text.

Why: we need raw page text to produce chunks and embeddings.

Expected output: a success message and the total number of pages.

Troubleshooting: ensure the `PDF_PATH` is correct and accessible; file path differences (local vs. remote runtime) are a common source of `FileNotFoundError`.

### Extraction From PDF

In [9]:
# EXTRACT TEXT FROM PDF

print("\n")
print("="*80)
print("EXTRACTING TEXT")
print("="*80)

full_text = ""

for page_number in range(len(document)):

    page = document[page_number]
    text = page.get_text()
    text = text.replace("\n", " ")
    text = " ".join(text.split())

    full_text += text + " "

print(" Text Extraction Completed!")
print(" Total Characters:", len(full_text))



EXTRACTING TEXT
 Text Extraction Completed!
 Total Characters: 39496


This cell iterates over each page and extracts the raw text, concatenating it into a single `full_text` string for downstream processing.

Why: extracting all text up front simplifies chunking and embedding steps.

Expected output: completion message and total character count.

Note: PDF extraction quality depends on the source PDF; scanned documents may require OCR prior to meaningful text extraction.

### Preview Document Text

In [10]:
# PREVIEW DOCUMENT TEXT

print("\n")
print("="*80)
print("DOCUMENT PREVIEW")
print("="*80)

print(full_text[:1500])



DOCUMENT PREVIEW
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convoluti

This cell displays a snippet of the extracted text so you can manually validate extraction quality (e.g., detect missing text, headers, or OCR artifacts).

Why: quick visual checks save time by catching extraction problems early.

Expected output: the first ~1500 characters of `full_text`. If the text looks empty or garbled, revisit the extraction step or consider OCR.

### Chunking

In [11]:
#TEXT CHUNKING

print("\n")
print("="*80)
print("CREATING TEXT CHUNKS")
print("="*80)

def chunk_text(text, chunk_size=700, overlap=150):

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks



CREATING TEXT CHUNKS


This cell defines `chunk_text`, a helper that splits long documents into fixed-size, overlapping chunks used for embedding and retrieval.

Why: chunking balances context size (small enough for embeddings and fast search) with overlap to preserve sentence continuity across chunk boundaries.

Expected behavior: a list of text chunks covering the full document.

Tuning: adjust `chunk_size` and `overlap` depending on model context window and retrieval granularity; too-large chunks reduce retrieval precision, too-small chunks increase index size.

In [12]:
# GENERATE CHUNKS
chunks = chunk_text(full_text)
print("Total Chunks Created:", len(chunks))

Total Chunks Created: 72


This cell runs the chunking function on `full_text` to produce the `chunks` list that will be embedded and indexed.

Expected output: printed total number of chunks.

Considerations: for very large documents you may want to stream-chunk and embed in batches to avoid high memory usage.

### Chunks Sample

In [13]:
# DISPLAY SAMPLE CHUNKS

print("\n")
print("="*80)
print("SAMPLE CHUNKS")
print("="*80)

for i in range(min(3, len(chunks))):

    print("\n")
    print("-"*80)
    print(f"CHUNK {i+1}")
    print("-"*80)

    print(chunks[i])




SAMPLE CHUNKS


--------------------------------------------------------------------------------
CHUNK 1
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural netw


--------------------------------------------------------------------------------
CHUNK 2
--------------------

This cell prints a few sample chunks so you can visually inspect how the text was split and verify there are no abrupt sentence cuts.

Why: ensures chunk boundaries are sensible and that overlap preserves context.

Expected output: a printed preview of up to three chunks; if chunks show partial or corrupted content, tweak chunking parameters or pre-clean the text.

### Embeddings

In [14]:
# GENERATE EMBEDDINGS

print("\n")
print("="*80)
print("GENERATING EMBEDDINGS")
print("="*80)

embeddings = embedding_model.encode(chunks, show_progress_bar=True)

print("Embeddings Generated Successfully!")
print("Embedding Shape:", embeddings.shape)



GENERATING EMBEDDINGS


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings Generated Successfully!
Embedding Shape: (72, 384)


This cell encodes every chunk into an embedding vector using the loaded sentence transformer and prints the resulting embedding array shape.

Why: embeddings are what the vector store uses to compute similarity between queries and document passages.

Expected output: progress bar during encoding and a final shape (num_chunks, embedding_dim).

Performance note: embedding large numbers of chunks can be slow or memory heavy—use batching or GPU-backed encoding where supported.

### Embeddings to ChromaDB

In [15]:
# STORE EMBEDDINGS IN CHROMADB

print("\n")
print("="*80)
print("STORING EMBEDDINGS IN CHROMADB")
print("="*80)

for index, chunk in enumerate(chunks):

    collection.add(
        ids=[str(index)],
        embeddings=[embeddings[index].tolist()],
        documents=[chunk]
    )

print(" All Chunks Stored Successfully!")



STORING EMBEDDINGS IN CHROMADB
 All Chunks Stored Successfully!


This cell writes each chunk, its embedding, and a stable ID into the ChromaDB collection so later queries can retrieve matching passages.

Why: persisting embeddings lets the notebook perform fast nearest-neighbor searches without re-encoding the document each time.

Expected output: confirmation that chunks were stored.

Notes: when indexing many chunks, consider batching `collection.add` calls for speed. Also include metadata (page numbers, offsets) if you want traceable source references later.

### Vector Search

In [16]:
# TEST VECTOR SEARCH

print("\n")
print("="*80)
print("TESTING VECTOR SEARCH")
print("="*80)

test_query = "What is the document about and what is the weight , key , query matrices ? "
query_embedding = embedding_model.encode([test_query])
results = collection.query(query_embeddings=query_embedding.tolist(),n_results=3)
retrieved_docs = results["documents"][0]

for i, doc in enumerate(retrieved_docs):
    print("\n")
    print("-"*80)
    print(f"RETRIEVED CHUNK {i+1}")
    print("-"*80)
    print(doc[:500])





TESTING VECTOR SEARCH


--------------------------------------------------------------------------------
RETRIEVED CHUNK 1
--------------------------------------------------------------------------------
set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum 3 Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key. 3.2.1 Scaled Dot-Product Attention We cal


--------------------------------------------------------------------------------
RETRIEVED CHUNK 2
--------------------------------------------------------------------------------
ention" (Figure 2). The input consists of queries and keys of dimension dk, and values of dimension dv. We comp

This cell performs a quick vector-search sanity check: it embeds a test query, runs a nearest-neighbor search in Chroma, and prints the top retrieved passages.

Why: validates the embedding→index→query pipeline before using retrieval in production prompts.

Expected output: a list of the top N retrieved chunk previews. If results are irrelevant, re-check embedding model selection, normalization, or chunk granularity.

### Retriver

In [17]:
# CREATE RETRIEVAL FUNCTION

def retrieve_relevant_context(query,top_k=3):

    query_embedding = embedding_model.encode([query])

    results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=top_k
)

    retrieved_chunks = results["documents"][0]
    context = "\n\n".join(retrieved_chunks)

    return context, results


This cell defines `retrieve_relevant_context`, a helper that converts a question into an embedding, queries the vector store, and returns the concatenated top-k document chunks plus raw retrieval metadata.

Why: packaging retrieval logic keeps the RAG prompt construction simple and consistent.

Expected behavior: returns the combined context text and a `results` object with distances and document IDs.

Tip: return or persist metadata (document id, page) alongside chunks to support traceable citations in answers.

### Hallucination Mode

In [18]:
#  WITHOUT RETRIEVAL (HALLUCINATION MODE)

def ask_without_retrieval(question):

    prompt = f""" Answer the following question.
    Question:
    {question}
    """

    output = generator(prompt)
    answer = output[0]["generated_text"].replace(prompt, "")

    return answer

This cell defines `ask_without_retrieval`, the baseline prompt flow that asks the model to answer using only its internal knowledge.

Why: this baseline reveals how the model behaves unaided by document context and highlights hallucination risk.

Expected output: a generated text answer that may or may not align with the PDF.

Use-case: useful for comparison with grounded answers; do not use for factual or auditable responses.

### With Retrieval Mode

In [19]:
# WITH RETRIEVAL (RAG MODE)

def ask_with_retrieval(question):

    context, retrieval_results = retrieve_relevant_context(question)
    prompt = f""" You are a helpful AI assistant. Use ONLY the context below to answer the question.
    CONTEXT:  {context}
    QUESTION:  {question}

    ANSWER:
    """

    output = generator(prompt)
    answer = output[0]["generated_text"].replace(prompt, "")
    return context, answer, retrieval_results


This cell defines `ask_with_retrieval`, the RAG-style flow that prepends only the retrieved document context to the question and asks the model to produce an answer grounded in that context.

Why: constraining the model with retrieved passages dramatically reduces hallucination and improves factual alignment with the source PDF.

Expected output: a tuple of (`context`, `answer`, `retrieval_results`).

Caveat: prompt engineering matters—be explicit that the model should use only the provided context and explicitly cite or indicate when information is missing.

### Input Question

In [22]:
# USER QUESTION INPUT

print("\n")
print("="*80)
print("DOCUMENT QUESTION ANSWERING")
print("="*80)

question = input(" Ask a question from the PDF: ")



DOCUMENT QUESTION ANSWERING
 Ask a question from the PDF: How  Transformer uses multi-head attention?


This cell prompts the user for an input question about the PDF and stores it in `question`.

Why: separating input collection keeps interactive steps explicit and reproducible.

Expected behavior: waits for user input; if running non-interactively, set `question` directly in the cell.

### Answer without reteival

In [23]:
# Generate answer without reteuval

print("\n")
print("="*100)
print("ANSWER WITHOUT RETRIEVAL (POSSIBLE HALLUCINATION)")
print("="*100)

hallucinated_answer = ask_without_retrieval(question)
print(textwrap.fill(hallucinated_answer,width=100))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




ANSWER WITHOUT RETRIEVAL (POSSIBLE HALLUCINATION)
     Answer:     In a Transformer model, multi-head attention is used to capture information from
different representation subspaces at different positions within the input sequence. This is
achieved by applying multiple attention heads in parallel to the same input query, key, and value
vectors. Each attention head computes a weighted sum of the input vectors based on the corresponding
attention scores, and the outputs from all attention heads are concatenated and passed through a
feed-forward network to produce the final output. This allows the model to learn and attend to
multiple contextual relationships simultaneously, improving its ability to capture long-range
dependencies and better understand the input sequence.


This cell runs the baseline (no-retrieval) generation to produce an answer using only the model's priors so you can compare it with the grounded response.

Why: comparing both outputs highlights the improvement provided by retrieval and surfaces hallucinations.

Expected output: the baseline answer printed to the notebook; if generation fails, check the `generator` pipeline and model device allocation.

### ANSWER WITH RETRIEVAL

In [24]:
# Generate the answer with retrieval
print("\n")
print("="*100)
print(" RETRIEVAL-AUGMENTED RESPONSE")
print("="*100)
context, grounded_answer, retrieval_results = ask_with_retrieval( question)
print("\n")
print("="*80)
print(" RETRIEVED CONTEXT")
print("="*80)
print(textwrap.fill(context[:2000],width=100))
print("\n")
print("="*80)
print(" GROUNDED ANSWER")
print("="*80)
print(textwrap.fill(grounded_answer,width=100))



Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




 RETRIEVAL-AUGMENTED RESPONSE


 RETRIEVED CONTEXT
ue to the reduced dimension of each head, the total computational cost is similar to that of single-
head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The
Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention"
layers, the queries come from the previous decoder layer, and the memory keys and values come from
the output of the encoder. This allows every position in the decoder to attend over all positions in
the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-
sequence models such as [38, 2, 9]. • The encoder contains self-attention layers. In a self-
attention layer all of the  itecture. The Transformer follows this overall architecture using
stacked self-attention and point-wise, fully connected layers for both the encoder and decoder,
shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and De

This cell runs the RAG flow: it retrieves the top-k context for the user question, calls the generator with that context, and prints both the retrieved passages and the grounded answer.

Why: this is the primary production flow for document-grounded QA—outputs here should be preferred for factual queries about the PDF.

Expected output: printed retrieved context snippet(s) and a grounded answer; if context is empty or irrelevant, verify embeddings and index correctness.

### SIMILARITY SCORE ANALYSIS

In [25]:
# Similarity Score Analysis

print("\n")
print("="*80)
print("SIMILARITY SCORE ANALYSIS")
print("="*80)

distances = retrieval_results["distances"][0]

for i, distance in enumerate(distances):
    similarity_score = 1 - distance
    print(f"Chunk {i+1} Similarity Score: {similarity_score:.4f}")



SIMILARITY SCORE ANALYSIS
Chunk 1 Similarity Score: 0.3916
Chunk 2 Similarity Score: 0.1379
Chunk 3 Similarity Score: 0.1256


This cell inspects the raw retrieval distances and converts them into similarity scores so you can judge how confidently the vector store matched each chunk to the query.

Why: similarity scores help decide whether the retrieved context is trustworthy or whether more retrieval tuning is needed.

Expected output: per-chunk similarity values printed; low scores indicate weak matches and may explain poor grounded answers.

### COMPARISON TABLE

In [26]:
# Comparison Table

print("\n")
print("="*80)
print("RAG VS HALLUCINATION COMPARISON")
print("="*80)

comparison = pd.DataFrame({
   "Approach": ["Without Retrieval", "With Retrieval"],
   "Knowledge Source": ["LLM Internal Memory", "Retrieved PDF Context"],
   "Hallucination Risk": ["High", "Low"],
  "Grounded Response": ["No", "Yes" ]
   })

print(comparison)



RAG VS HALLUCINATION COMPARISON
            Approach       Knowledge Source Hallucination Risk  \
0  Without Retrieval    LLM Internal Memory               High   
1     With Retrieval  Retrieved PDF Context                Low   

  Grounded Response  
0                No  
1               Yes  


In [3]:
!pip install chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [4]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")

In [5]:
collection = client.create_collection(name="Students")

In [6]:
student_info = """
Alexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,
is a member of the programming and chess clubs who enjoys pizza, swimming, and hiking
in her free time in hopes of working at a tech company after graduating from the University of Washington.
"""

club_info = """
The university chess club provides an outlet for students to come together and enjoy playing
the classic strategy game of chess. Members of all skill levels are welcome, from beginners learning
the rules to experienced tournament players. The club typically meets a few times per week to play casual games,
participate in tournaments, analyze famous chess matches, and improve members' skills.
"""

university_info = """
The University of Washington, founded in 1861 in Seattle, is a public research university
with over 45,000 students across three campuses in Seattle, Tacoma, and Bothell.
As the flagship institution of the six public universities in Washington state,
UW encompasses over 500 buildings and 20 million square feet of space,
including one of the largest library systems in the world.
"""

In [7]:
collection.add(
    documents = [student_info, club_info, university_info],
    metadatas = [{"source": "student info"},{"source": "club info"},{'source':'university info'}],
    ids = ["id1", "id2", "id3"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.9MiB/s]


In [8]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2
)

results

{'ids': [['id1', 'id2']],
 'embeddings': None,
 'documents': [['\nAlexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,\nis a member of the programming and chess clubs who enjoys pizza, swimming, and hiking\nin her free time in hopes of working at a tech company after graduating from the University of Washington.\n',
   "\nThe university chess club provides an outlet for students to come together and enjoy playing\nthe classic strategy game of chess. Members of all skill levels are welcome, from beginners learning\nthe rules to experienced tournament players. The club typically meets a few times per week to play casual games,\nparticipate in tournaments, analyze famous chess matches, and improve members' skills.\n"]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'student info'}, {'source': 'club info'}]],
 'distances': [[1.2946667671203613, 1.395403265953064]]}

In [ ]:
from chromadb.utils import embedding_functions

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key="YOUR_OPENAI_API_KEY",
    model_name="text-embedding-3-small"
)
students_embeddings = openai_ef([student_info, club_info, university_info])
print(students_embeddings)

[array([ 0.01264954, -0.0340271 ,  0.04516602, ..., -0.01287079,
        0.037323  ,  0.00566864], dtype=float32), array([-4.84085083e-03, -7.15851784e-05,  2.42309570e-02, ...,
        6.46209717e-03, -2.38342285e-02, -1.12838745e-02], dtype=float32), array([-0.01256561, -0.05831909,  0.07293701, ..., -0.0125885 ,
       -0.01064301, -0.00351524], dtype=float32)]
